# DATA_Source

In [1]:
import pandas as pd

files = ['agios_2021.xlsx', 'agios_2022.xlsx', 'agios_2023.xlsx', 'agios_2024.xlsx']
df_list = [pd.read_excel(file) for file in files]
df = pd.concat(df_list, ignore_index=True)

plan_comptable = pd.read_excel('Plan comptable.xlsx')
dim_client = pd.read_excel('A02_01 Tables Dimension Client.xlsx', sheet_name='Dim Client')
table_agios = pd.read_excel('TABLE_AGIOS.xlsx')

# Test Mois_Manquants

In [22]:
df_merged = df.merge(plan_comptable[['Compte', 'ID_Tiers', 'Intitule compte']], left_on='NUM_CMPT', right_on='Compte', how='left') 
df_merged = df_merged.merge(dim_client[['ID_Client', 'Type Client']], left_on='ID_Tiers', right_on='ID_Client', how='left')
df_merged = df_merged.merge(table_agios[['ID_Commission', 'Nature']], left_on='ID_Commission', right_on='ID_Commission', how='left')

df_filtered = df_merged[df_merged['Nature'] == 'Frais de tenue de compte']

df_filtered['DATE'] = pd.to_datetime(df_filtered['DATE'], errors='coerce')
df_filtered['YearMonth'] = df_filtered['DATE'].dt.to_period('M')

df_filtered = df_filtered[(df_filtered['DATE'] >= '2021-01-29') & (df_filtered['DATE'] <= '2024-11-29')]

pivot_table = pd.pivot_table(df_filtered, values='MONTANT', index=['ID_Client', 'Compte'], columns='YearMonth', aggfunc='sum')

anomalies = []

for (client_id, compte), row in pivot_table.iterrows():
    valid_dates = row.dropna().index
    
    if len(valid_dates) > 0:
        first_date = valid_dates.min()
        last_date = valid_dates.max()
        
        expected_months = pd.period_range(start=first_date, end=last_date, freq='M')
        
        missing_months = expected_months.difference(valid_dates)
        
        if len(missing_months) > 0:
            intitule_compte = df_merged.loc[df_merged['ID_Client'] == client_id, 'Intitule compte'].iloc[0]
            type_client = df_merged.loc[df_merged['ID_Client'] == client_id, 'Type Client'].iloc[0]
            
            anomalies.append({
                'Client_ID': client_id,
                'Compte': compte,
                'Intitule_Compte': intitule_compte,
                'Type_Client': type_client,
                'Missing_Months': ', '.join(str(m) for m in missing_months)
            })

anomalies_df = pd.DataFrame(anomalies)

anomalies_df.to_excel('Test_MoisManquants.xlsx', index=False)

C:\Users\rbelhaj\AppData\Local\Temp\ipykernel_5868\530640274.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered['DATE'] = pd.to_datetime(df_filtered['DATE'], errors='coerce')
C:\Users\rbelhaj\AppData\Local\Temp\ipykernel_5868\530640274.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered['YearMonth'] = df_filtered['DATE'].dt.to_period('M')


# Test Frais_Trimestriels

In [14]:
anomalies_trimestriels = []

for (client_id, compte), row in pivot_table.iterrows():
    valid_dates = row.dropna().index
    
    mois_valides = [str(d.month).zfill(2) for d in valid_dates]
    if all(mois in ['03', '06', '09', '12'] for mois in mois_valides):
        intitule_compte = plan_comptable.loc[plan_comptable['Compte'] == compte, 'Intitule compte'].iloc[0]
        type_client = dim_client.loc[dim_client['ID_Client'] == client_id, 'Type Client'].iloc[0]
        
        anomalies_trimestriels.append({
            'Client_ID': client_id,
            'Compte': compte,
            'Intitule_Compte': intitule_compte,
            'Type_Client': type_client,
            'Frais_Trimestriels': ', '.join(str(d) for d in valid_dates)
        })


anomalies_trimestriels_df = pd.DataFrame(anomalies_trimestriels)

anomalies_trimestriels_df.to_excel('Test_FraisTrimestriels.xlsx', index=False)

# Test Montant_Different

In [16]:
anomalies_montants = []

for (client_id, compte), row in pivot_table.iterrows():
    valid_amounts = row.dropna().unique()
    
    if len(valid_amounts) > 1:
        intitule_compte = plan_comptable.loc[plan_comptable['Compte'] == compte, 'Intitule compte'].iloc[0]
        type_client = dim_client.loc[dim_client['ID_Client'] == client_id, 'Type Client'].iloc[0]
        
        anomalies_montants.append({
            'Client_ID': client_id,
            'Compte': compte,
            'Intitule_Compte': intitule_compte,
            'Type_Client': type_client,
            'Different_Amounts': ', '.join(str(amount) for amount in valid_amounts)
        })

anomalies_montants_df = pd.DataFrame(anomalies_montants)
anomalies_montants_df.to_excel('Test_Montants_Differents.xlsx', index=False)

# Test Montant selon Type_Client

In [20]:
anomalies_montants = []

for (client_id, compte), row in pivot_table.iterrows():
    valid_amounts = row.dropna().unique()
    
    intitule_compte = plan_comptable.loc[plan_comptable['Compte'] == compte, 'Intitule compte'].iloc[0]
    type_client = dim_client.loc[dim_client['ID_Client'] == client_id, 'Type Client'].iloc[0]
    
    if type_client == 1:
        tarification = 60
    elif type_client == 2:
        tarification = 300
    elif 'ONG' in intitule_compte:
        tarification = 200
    else:
        tarification = None
    
    if tarification is not None and len(valid_amounts) > 1:
        anomalies_montants.append({
            'Client_ID': client_id,
            'Compte': compte,
            'Intitule_Compte': intitule_compte,
            'Type_Client': type_client,
            'Tarification': tarification,
            'Different_Amounts': ', '.join(str(amount) for amount in valid_amounts)
        })

anomalies_montants_df = pd.DataFrame(anomalies_montants)
anomalies_montants_df.to_excel('Test_Tarification.xlsx', index=False)

C:\Users\rbelhaj\AppData\Local\Temp\ipykernel_5868\2153961437.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered['DATE'] = pd.to_datetime(df_filtered['DATE'], errors='coerce')
C:\Users\rbelhaj\AppData\Local\Temp\ipykernel_5868\2153961437.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered['YearMonth'] = df_filtered['DATE'].dt.to_period('M')
